# RetinaNet model for snowpole detection with LiDar and RGB datasets

### Import Packages

In [ ]:
from pathlib import Path
import torchvision
import torch
import torch.optim as optim
from torchvision.models.detection import RetinaNet_ResNet50_FPN_V2_Weights
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
import pandas as pd
from torch.utils.data import Dataset
from PIL import Image # Or import cv2 if using OpenCV
import os
from transform import get_transforms, collate_fn
from dataloader import LidarDataset
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import time
import torchmetrics
from torchmetrics.detection import MeanAveragePrecision
import torchvision.transforms as T

In [ ]:
!pip install albumentations
!pip install torchmetrics
!pip install 'typing_extensions>=4.10'
!pip install --force-reinstall numpy pandas
!pip install --force-reinstall 'numpy<2.0'
!pip install pycocotools

In [ ]:
# Create symlinks for RGB
!mkdir -p data/rgb/images/train data/rgb/labels/train
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/images/train/* data/rgb/images/train/
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/labels/train/* data/rgb/labels/train/

!mkdir -p data/rgb/images/valid data/rgb/labels/valid
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/images/valid/* data/rgb/images/valid/
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/labels/valid/* data/rgb/labels/valid/

!mkdir -p data/rgb/images/test data/rgb/labels/test
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/images/test/* data/rgb/images/test/

# Create symlinks for LIDAR
!mkdir -p data/lidar/images/train data/lidar/labels/train
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/combined_color/train/* data/lidar/images/train/
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/labels/train/* data/lidar/labels/train/

!mkdir -p data/lidar/images/valid data/lidar/labels/valid
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/combined_color/valid/* data/lidar/images/valid/
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/labels/valid/* data/lidar/labels/valid/

!mkdir -p data/lidar/images/test data/lidar/labels/test
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/combined_color/test/* data/lidar/images/test/

In [ ]:
def convert_yolo_to_retinanet_csv(image_dir, label_dir, output_csv, class_csv, class_map):
    image_dir = Path(image_dir)
    label_dir = Path(label_dir)

    annotations = []

    for label_file in sorted(label_dir.glob('*.txt')):
        img_file = image_dir / (label_file.stem + '.PNG')
        if not img_file.exists():
            img_file = image_dir / (label_file.stem + '.png')
        if not img_file.exists():
            print(f"Image for {label_file.stem} not found, skipping.")
            continue

        with Image.open(img_file) as img:
            img_w, img_h = img.size

        with open(label_file, 'r') as f:
            lines = f.readlines()

        if not lines:
            annotations.append([str(img_file), '', '', '', '', ''])
            continue

        for line in lines:
            class_id, x_center, y_center, width, height = map(float, line.strip().split())
            class_id = int(class_id)
            class_name = class_map.get(class_id, f'class_{class_id}')
            x1 = (x_center - width / 2) * img_w
            y1 = (y_center - height / 2) * img_h
            x2 = (x_center + width / 2) * img_w
            y2 = (y_center + height / 2) * img_h
            annotations.append([str(img_file), int(x1), int(y1), int(x2), int(y2), class_name])

    # Save annotations CSV
    df = pd.DataFrame(annotations, columns=['image_path', 'x1', 'y1', 'x2', 'y2', 'class_name'])
    df.to_csv(output_csv, index=False, header=False)
    print(f"Saved {len(df)} annotations to {output_csv}")

    # Save class mapping CSV
    with open(class_csv, 'w') as f:
        for i, name in sorted(class_map.items()):
            f.write(f"{name},{i}\n")
    print(f"Saved class mapping to {class_csv}")



In [ ]:
# Example usage
class_map = {
    0: 'pole',
    # Add more classes here if needed
}

convert_yolo_to_retinanet_csv(
    image_dir='data/rgb/images/train',
    label_dir='data/rgb/labels/train',
    output_csv='annotations/rgb_train_annotations.csv',
    class_csv='classes.csv',
    class_map=class_map
)

convert_yolo_to_retinanet_csv(
    image_dir='data/rgb/images/valid',
    label_dir='data/rgb/labels/valid',
    output_csv='annotations/rgb_valid_annotations.csv',
    class_csv='classes.csv',
    class_map=class_map
)

convert_yolo_to_retinanet_csv(
    image_dir='data/rgb/images/test',
    label_dir='data/rgb/labels/test',
    output_csv='annotations/rgb_test_annotations.csv',
    class_csv='classes.csv',
    class_map=class_map
)

convert_yolo_to_retinanet_csv(
    image_dir='data/lidar/images/train',
    label_dir='data/lidar/labels/train',
    output_csv='annotations/lidar_train_annotations.csv',
    class_csv='classes.csv',
    class_map=class_map
)

convert_yolo_to_retinanet_csv(
    image_dir='data/lidar/images/valid',
    label_dir='data/lidar/labels/valid',
    output_csv='annotations/lidar_valid_annotations.csv',
    class_csv='classes.csv',
    class_map=class_map
)

convert_yolo_to_retinanet_csv(
    image_dir='data/lidar/images/test',
    label_dir='data/lidar/labels/test',
    output_csv='annotations/lidar_test_annotations.csv',
    class_csv='classes.csv',
    class_map=class_map
)


### 1. Load the pre-trained model (using V2 weights, which are often better)

In [ ]:
weights = RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT # Loads COCO pre-trained weights
model = torchvision.models.detection.retinanet_resnet50_fpn_v2(weights=weights)

### --- Modification for Custom Dataset ---
#### Get the number of input features for the classifier

In [ ]:
num_anchors = model.head.classification_head.num_anchors
in_channels = model.backbone.out_channels

#### Define your number of classes (e.g., number of classes in your custom dataset + 1 for background)
 IMPORTANT: Torchvision RetinaNet usually expects num_classes = actual_classes + 1 (for background)
 However, documentation suggests the head should be built with num_classes = actual_classes * num_anchors.
 Double-check the specific documentation for the version you use, but often it's handled like this:

In [ ]:
num_classes_custom = 2 # Replace with the actual number of classes in your dataset

In [ ]:
new_cls_head = RetinaNetClassificationHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=num_classes_custom
)

In [ ]:
# Replace the pre-trained head with the new one
model.head.classification_head = new_cls_head

# --- End Modification ---

In [ ]:
print("Current working directory:", os.getcwd())
# --- 1. Define Configuration ---
BATCH_SIZE = 16
#ANNOTATIONS_DIR = "/work/mathiamt/SnowPoleDetection/torchVision/pytorch-retinanet/annotations"
#CLASSES_FILE = "/work/mathiamt/SnowPoleDetection/torchVision/pytorch-retinanet/classes.csv"
ANNOTATIONS_DIR = "/work/chrsjoha/SnowConeDetection/torchVision/pytorch-retinanet/annotations"
CLASSES_FILE = "/work/chrsjoha/SnowConeDetection/torchVision/pytorch-retinanet/classes.csv"
# Define image directory - IMPORTANT: adjust this based on paths in your CSV
# If paths in CSV are like 'retinanet_data/lidar/images/train/image_1.png' and you run from project root:
IMG_DIR = '.'
# If paths in CSV are just 'image_1.png', use the full path to the specific train/valid directories
# IMG_DIR_TRAIN = '/work/mathiamt/SnowPoleDetection/torchVision/pytorch-retinanet/retinanet_data/lidar/images/train'
# IMG_DIR_VALID = '/work/mathiamt/SnowPoleDetection/torchVision/pytorch-retinanet/retinanet_data/lidar/images/valid'

# Get the transform functions
train_transforms = get_transforms(is_train=True)
val_transforms = get_transforms(is_train=False)

# --- 2. Create Dataset Instances ---
train_annotations_file = os.path.join(ANNOTATIONS_DIR, "rgb_train_annotations.csv")
val_annotations_file = os.path.join(ANNOTATIONS_DIR, "rgb_valid_annotations.csv")

# Make sure to provide all required arguments to LidarDataset:
# annotations_file, classes_file, img_dir, transforms
train_dataset = LidarDataset(
    annotations_file=train_annotations_file,
    classes_file=CLASSES_FILE,
    img_dir=None, # Or IMG_DIR_TRAIN if paths in CSV are just filenames
    transforms=train_transforms
)

val_dataset = LidarDataset(
    annotations_file=val_annotations_file,
    classes_file=CLASSES_FILE,
    img_dir=None, # Or IMG_DIR_VALID if paths in CSV are just filenames
    transforms=val_transforms
)

# --- 3. Create DataLoader Instances ---
# Now pass the dataset instances to the DataLoader
train_loader = DataLoader(
    dataset=train_dataset, # Pass the instantiated dataset
    batch_size=BATCH_SIZE,
    shuffle=True,          # Shuffle for training
    num_workers=2,         # Adjust based on your system
    collate_fn=collate_fn
)

val_loader = DataLoader(
    dataset=val_dataset,   # Pass the instantiated dataset
    batch_size=BATCH_SIZE,
    shuffle=False,         # No shuffle for validation
    num_workers=2,
    collate_fn=collate_fn
)

# --- Ready for Training ---
print(f"Created train_loader with {len(train_loader)} batches.")
print(f"Created val_loader with {len(val_loader)} batches.")

# You can now use train_loader and val_loader in your training loop

##### --- 1. Define the Model ---

In [ ]:
# --- 2. Set the Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model.to(device) # Move model to the chosen device




In [ ]:
# --- 3. Define the Optimizer ---
# Get parameters that require gradients
params = [p for p in model.parameters() if p.requires_grad]

In [ ]:
# Choose optimizer and learning rate
learning_rate = 0.001
# optimizer = optim.Adam(params, lr=learning_rate, weight_decay=0.0001)
optimizer = optim.SGD(params, lr=learning_rate, momentum=0.9, weight_decay=0.0005) # SGD often used in papers

print("Model, Device, and Optimizer are set up.")

In [63]:

# --- Training Configuration ---
num_epochs = 100 # Or choose desired number of epochs
output_dir = "checkpoints"
os.makedirs(output_dir, exist_ok=True) # Create directory if it doesn't exist
best_model_path = os.path.join(output_dir, "retinanet_best_map50.pth")
last_model_path = os.path.join(output_dir, "retinanet_last_epoch.pth")
# Optional: Learning Rate Scheduler
# Example: Reduce LR by a factor of 0.1 every 3 epochs
# scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# --- Instantiate the Metric ---
# We will calculate mAP using the COCO standard evaluation procedure
# It provides map, map_50, map_75, map_small, map_medium, map_large
metric = MeanAveragePrecision(iou_type="bbox")
# Move metric state to the correct device (important if using GPU)
metric.to(device)

best_map_50 = 0.0 # Initialize best mAP@0.95 score

# Lists to store history (optional)
train_loss_history = []
map_history = [] # Store mAP (0.50:0.95)
map_50_history = [] # Store mAP (0.50)

print("Starting Training...")
start_time = time.time()

for epoch in range(num_epochs):
    # --- Training Phase ---
    model.train() # Set model to training mode
    running_loss = 0.0
    epoch_train_loss = 0.0
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 10)

    # Iterate over data. Add tqdm here for a progress bar if desired:
    # from tqdm.notebook import tqdm
    # for batch_idx, (images, targets) in enumerate(tqdm(train_loader, desc=f"Training Epoch {epoch+1}")):
    for batch_idx, (images, targets) in enumerate(train_loader):
        # Move data to the correct device
        images = list(image.to(device) for image in images)
        # Targets is a list of dicts. Move tensors inside each dict to device.
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        # Torchvision detection models return a dict of losses during training
        loss_dict = model(images, targets)

        # Sum up the losses
        losses = sum(loss for loss in loss_dict.values())

        # Check for invalid loss values (NaN or Inf)
        if not torch.isfinite(losses):
            print(f"WARNING: Non-finite loss detected: {losses.item()}. Skipping batch {batch_idx}.")
            # Optionally: print loss_dict to investigate individual losses
            # print(loss_dict)
            continue # Skip optimizer step if loss is invalid

        # Backward pass
        losses.backward()

        # Optimizer step (update weights)
        optimizer.step()

        # --- Statistics ---
        current_loss = losses.item()
        running_loss += current_loss
        epoch_train_loss += current_loss

        # Print loss statistics every N batches (e.g., every 10 batches)
        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx+1}/{len(train_loader)} - Loss: {current_loss:.4f} (Avg: {running_loss/10:.4f})")
            # You can also print individual losses from loss_dict if needed:
            # loss_str = " ".join([f"{k}: {v.item():.4f}" for k, v in loss_dict.items()])
            # print(f"     Loss details: {loss_str}")
            running_loss = 0.0 # Reset running loss for the next N batches

    # Calculate average training loss for the epoch
    avg_epoch_train_loss = epoch_train_loss / len(train_loader)
    train_loss_history.append(avg_epoch_train_loss)
    print(f"Epoch {epoch+1} Training Loss: {avg_epoch_train_loss:.4f}")

    # Optional: Update the learning rate
    # if scheduler:
    #     scheduler.step()

    # --- Validation Phase ---
    print("\nRunning Validation and Calculating mAP...")
    model.eval() 
    with torch.no_grad(): 
        for images, targets in val_loader: # We still load targets for potential future evaluation
            images = list(image.to(device) for image in images)
           
            targets_cpu = [{k: v.cpu() for k, v in t.items()} for t in targets]
            
            # Get model predictions
            outputs = model(images)
            # Move predictions to CPU
            outputs_cpu = [{k: v.cpu() for k, v in t.items()} for t in outputs]

            # Update the metric state
            # Expected format: List[Dict[str, Tensor]] for both preds and targets
            # preds dict keys: 'boxes', 'scores', 'labels'
            # target dict keys: 'boxes', 'labels'
            metric.update(outputs_cpu, targets_cpu)

    # Compute the results over all batches
    map_50 = 0.0 # Default value in case of error
    map_val = 0.0
    try:
        results = metric.compute()
        map_50 = results['map_50'].item()
       
        map_val = results['map'].item() 
        map_history.append(map_val)
        map_50_history.append(map_50)

        print(f"Epoch {epoch+1} Validation Results:")
        print(f"  mAP@.50-.95: {map_val:.4f}")
        print(f"  mAP@.50:     {map_50:.4f}")


    except Exception as e:
        print(f"Could not compute mAP for epoch {epoch+1}: {e}")
        # Append placeholder values if computation failed
        map_history.append(0.0)
        map_50_history.append(0.0)


    # Reset the metric for the next epoch
    metric.reset()

    # --- Save Checkpoint Logic ---
    print(f"*** {map_50}: {best_map_50} ***")
    if map_50 > best_map_50:
        best_map_50 = map_50
        torch.save(model.state_dict(), best_model_path)
        print(f"*** New best model saved based on mAP@0.50: {best_map_50:.4f} at epoch {epoch+1} ***")
        print(f"*** Saved to: {best_model_path} ***")

    # Optionally, save the model from the very last epoch regardless of performance
    torch.save(model.state_dict(), last_model_path)
    print(f"Saved last epoch model state to: {last_model_path}") # Uncomment if needed


    print("Validation Phase Complete for Epoch", epoch+1)



# --- Training Complete ---
end_time = time.time()
total_time = end_time - start_time
print(f"\nTraining finished in {total_time // 60:.0f}m {total_time % 60:.0f}s")


#Optional: Plot training loss and mAP

fig, ax1 = plt.subplots()
color = 'tab:red'
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss', color=color)
ax1.plot(range(1, num_epochs + 1), train_loss_history, color=color)
ax1.tick_params(axis='y', labelcolor=color)
ax2 = ax1.twinx() # instantiate a second axes that shares the same x-axis
color = 'tab:blue'
ax2.set_ylabel('mAP@.50', color=color) # we already handled the x-label with ax1
ax2.plot(range(1, num_epochs + 1), map_50_history, color=color, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color)
fig.tight_layout() # otherwise the right y-label is slightly clipped
plt.title('Training Loss and mAP@.50 Over Epochs')
plt.show()

Starting Training...

Epoch 1/100
----------
  Batch 10/21 - Loss: 0.1774 (Avg: 0.1877)
  Batch 20/21 - Loss: 0.1968 (Avg: 0.1850)
Epoch 1 Training Loss: 0.1843

Running Validation and Calculating mAP...
Epoch 1 Validation Results:
  mAP@.50-.95: 0.3083
  mAP@.50:     0.8141
*** 0.8141121864318848: 0.0 ***
*** New best model saved based on mAP@0.50: 0.8141 at epoch 1 ***
*** Saved to: checkpoints/retinanet_best_map50.pth ***
Saved last epoch model state to: checkpoints/retinanet_last_epoch.pth
Validation Phase Complete for Epoch 1

Epoch 2/100
----------
  Batch 10/21 - Loss: 0.1839 (Avg: 0.1811)
  Batch 20/21 - Loss: 0.2230 (Avg: 0.1904)
Epoch 2 Training Loss: 0.1987

Running Validation and Calculating mAP...
Epoch 2 Validation Results:
  mAP@.50-.95: 0.3124
  mAP@.50:     0.8178
*** 0.8177618384361267: 0.8141121864318848 ***
*** New best model saved based on mAP@0.50: 0.8178 at epoch 2 ***
*** Saved to: checkpoints/retinanet_best_map50.pth ***
Saved last epoch model state to: checkpo

KeyboardInterrupt: 

## Detection on test data
Loading trained weights

In [ ]:
best_path = 'checkpoints/retinanet_best_map50.pth'
num_classes_custom = 2

weights = torchvision.models.detection.RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT
model = torchvision.models.detection.retinanet_resnet50_fpn_v2(weights=None)

num_anchors = model.head.classification_head.num_anchors
in_channels = model.backbone.out_channels
new_cls_head = torchvision.models.detection.retinanet.RetinaNetClassificationHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=num_classes_custom
)
model.head.classification_head = new_cls_head

model.load_state_dict(torch.load(best_path))
model.eval()

state = torch.load(best_path)
missing, unexpected = model.load_state_dict(state, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model loaded and ready for inference.")

## Function for transforming test images
transform.py can not be used here as it expects image and bounding boxes, but we only have images for the tests

In [ ]:
def preprocess_image(image_path, img_size):
    image = Image.open(image_path).convert("RGB")
    transform = T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225))
    ])
    image_tensor = transform(image).unsqueeze(0)
    return image, image_tensor

def get_next_exp_folder(base_parent="detections_txt", base_name="exp"):
    os.makedirs(base_parent, exist_ok=True)
    i = 1
    while os.path.exists(os.path.join(base_parent, f"{base_name}{i}")):
        i += 1
    return os.path.join(base_parent, f"{base_name}{i}")


In [ ]:
import glob

def convert_to_yolo_format(box, score, label, image_width, image_height):
    """
    Converts a box from [x1, y1, x2, y2] to YOLO format:
    class_id confidence center_x center_y width height 
    """
    x1, y1, x2, y2 = box
    box_width = x2 - x1
    box_height = y2 - y1
    center_x = x1 + box_width / 2
    center_y = y1 + box_height / 2

    # Normalize
    center_x /= image_width
    center_y /= image_height
    box_width /= image_width
    box_height /= image_height

    return f"{label} {center_x:.6f} {center_y:.6f} {box_width:.6f} {box_height:.6f} {score:.6f}"




In [ ]:
import cv2
import numpy as np
from PIL import Image

def plot_and_save_detections(image, boxes, scores, labels, save_path, class_name="pole", score_threshold=0.05):
    """
    Draws bounding boxes and saves the image.
    """
    image_np = np.array(image).copy()

    for box, score, label in zip(boxes, scores, labels):
        if score < score_threshold or label != 0:
            continue  # Only plot poles with sufficient score

        x1, y1, x2, y2 = map(int, box)

        # Draw rectangle
        cv2.rectangle(image_np, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Label text
        text = f"{class_name} {score:.2f}"
        cv2.putText(image_np, text, (x1, max(y1 - 10, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Convert back to PIL and save
    vis_image = Image.fromarray(image_np)
    vis_image.save(save_path)




In [ ]:
# --- Parameters ---
score_threshold = 0.05
output_folder = get_next_exp_folder("detections_txt")
os.makedirs(output_folder, exist_ok=True)

# --- Get images (both .png and .PNG) ---
test_folder = "/datasets/tdt4265/ad/open/Poles/rgb/images/test/"
#test_folder = "/datasets/tdt4265/ad/open/Poles/lidar/combined_color/test/"
image_paths = glob.glob(os.path.join(test_folder, "*.png")) + glob.glob(os.path.join(test_folder, "*.PNG"))

print(f"Found {len(image_paths)} test images.")

# --- Process each image ---
for image_path in image_paths:
    original_image, input_tensor = preprocess_image(image_path, 1024)
    input_tensor = input_tensor.to(device)

    #image_width, image_height = original_image.size
    #print(f"Width: {image_width}, Height: {image_height}")
    input_width, input_height = input_tensor.shape[-1], input_tensor.shape[-2]  # PyTorch tensor shape is (C, H, W)
    print(f"Width: {input_width}, Height: {input_height}")
    
    with torch.no_grad():
        outputs = model(input_tensor)

    detections = outputs[0]
    boxes = detections['boxes'].cpu().numpy()
    scores = detections['scores'].cpu().numpy()
    labels = detections['labels'].cpu().numpy()

    detections_found = 0
    yolo_lines = []

    for box, score, label in zip(boxes, scores, labels):
        if score >= score_threshold:
            yolo_line = convert_to_yolo_format(box, score, label, input_width, input_height)
            yolo_lines.append(yolo_line)
            detections_found += 1

    if detections_found > 0:
        base_filename = os.path.basename(image_path)
        filename_wo_ext = os.path.splitext(base_filename)[0]

        # Use correct naming
        txt_filename = f"{filename_wo_ext}.txt"
    
        txt_path = os.path.join(output_folder, txt_filename)

        with open(txt_path, "w") as f:
            for line in yolo_lines:
                f.write(line + "\n")
        
        #vis_save_path = os.path.join(vis_folder, f"{filename_wo_ext}.png")
        #plot_and_save_detections(original_image, boxes, scores, labels, vis_save_path, score_threshold=score_threshold)


        print(f"{filename_wo_ext}: {detections_found} pole detections saved.")
    else:
        print(f"{os.path.basename(image_path)}: No detections (no file written).")

print("Detection and export completed for all images.")

# --- Visualization output folder ---
#vis_folder = os.path.join(output_folder, "vis")
#os.makedirs(vis_folder, exist_ok=True)

## Drawing boxes on a image

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def plot_detections_on_image(image, boxes, scores, labels, class_names, score_threshold=0.5):
    """
    Draws bounding boxes and labels on the image.

    Args:
        image (PIL.Image): The original image.
        boxes (array): Bounding boxes in [x1, y1, x2, y2] format.
        scores (array): Confidence scores.
        labels (array): Class IDs.
        class_names (list): Class names where index matches label ID.
        score_threshold (float): Minimum score to display a box.
    """
    # Convert PIL image to numpy array for OpenCV
    image_np = np.array(image).copy()

    for box, score, label in zip(boxes, scores, labels):
        if score < score_threshold:
            continue  # Skip low-confidence detections

        x1, y1, x2, y2 = [int(coord) for coord in box]

        # Draw rectangle
        cv2.rectangle(image_np, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Put label text
        label_text = f"{class_names[label]}: {score:.2f}"
        cv2.putText(image_np, label_text, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    plt.figure(figsize=(10, 10))
    plt.imshow(image_np)
    plt.axis('off')
    plt.show()
class_names = ["background", "pole"] 


In [ ]:

#image_path = "/datasets/tdt4265/ad/open/Poles/lidar/combined_color/test/image_0.png" # Change this to your test image path
image_path = "/datasets/tdt4265/ad/open/Poles/rgb/images/test/frame_000315.PNG" # Change this to your test image path
original_image, input_tensor = preprocess_image(image_path, 640)
input_tensor = input_tensor.to(device)
with torch.no_grad():
    outputs = model(input_tensor)

# Outputs is a list of detections (since we passed a batch of 1 image, outputs[0])
detections = outputs[0]

boxes = detections['boxes'].cpu().numpy()
scores = detections['scores'].cpu().numpy()
labels = detections['labels'].cpu().numpy()


In [ ]:
# Set the score threshold
score_threshold = 0.015

# Count detections above the threshold
num_detections = sum(score >= score_threshold for score in scores)

print(f"Detections found with score ≥ {score_threshold}: {num_detections}")


plot_detections_on_image(original_image, boxes, scores, labels, class_names, score_threshold=0.05)

In [ ]:
import pandas as pd

power_path = '/work/chrsjoha/SnowConeDetection/power_log_retina.csv'

df = pd.read_csv(power_path, header=None, names=['timestamp', 'power'])

# Remove the " W" and convert to float
df['power'] = df['power'].str.replace(' W', '', regex=False).astype(float)

average_power = df['power'].mean()

print(f'Average Power: {average_power:.2f} W')
